# Creating Sentence Embeddings

Different from the seminal paper [Word2Vec](https://arxiv.org/abs/1301.3781) that introduced vectorized embeddings for **words** to do word similarity tasks, sentence embeddings deal with **sentences** and use the [transformer architecture](https://huggingface.co/sentence-transformers) to do so.

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))
train_dataset = train_dataset.remove_columns(["idx"])

In [3]:
train_dataset[5]

{'premise': "my walkman broke so i'm upset now i just have to turn the stereo up real loud",
 'hypothesis': "I'm upset that my walkman broke and now I have to turn the stereo up really loud.",
 'label': 0}

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Getting base models
embedding_model = SentenceTransformer('bert-base-uncased')
embedding_model_two = SentenceTransformer('microsoft/mpnet-base')

In [6]:
#BertModel
embedding_model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [7]:
#MPNetModel
embedding_model_two

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

# SoftMaxLoss

In [8]:
from sentence_transformers import losses
# SoftmaxLoss for MPNetModel
train_loss = losses.SoftmaxLoss(model=embedding_model_two, sentence_embedding_dimension=embedding_model_two.get_sentence_embedding_dimension(), num_labels=3)

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for STSB
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

In [10]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mpnet_embedding_model_softmax",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

In [11]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model_two,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ichglaubeya (ichglaubeya-myself) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.077200
200,0.989800
300,0.941100
400,0.901700
500,0.877800
600,0.882100
700,0.870300
800,0.852000
900,0.833600
1000,0.837500


TrainOutput(global_step=1563, training_loss=0.8691243094202042, metrics={'train_runtime': 426.6231, 'train_samples_per_second': 117.199, 'train_steps_per_second': 3.664, 'total_flos': 0.0, 'train_loss': 0.8691243094202042, 'epoch': 1.0})

In [12]:
evaluator(embedding_model_two)

{'pearson_cosine': np.float64(0.33616042317357014),
 'spearman_cosine': np.float64(0.41535162612704)}

In [13]:
train_loss = losses.SoftmaxLoss(model=embedding_model, sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(), num_labels=3)
# Define the training arguments
args_one = SentenceTransformerTrainingArguments(
    output_dir="bert_embedding_model_softmax",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args_one,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.080000
200,0.947700
300,0.893000
400,0.846100
500,0.840300
600,0.828800
700,0.815000
800,0.800800
900,0.774300
1000,0.766300


TrainOutput(global_step=1563, training_loss=0.8175519966423245, metrics={'train_runtime': 158.9972, 'train_samples_per_second': 314.471, 'train_steps_per_second': 9.83, 'total_flos': 0.0, 'train_loss': 0.8175519966423245, 'epoch': 1.0})

In [14]:
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.5447547055531128),
 'spearman_cosine': np.float64(0.6102299116869513)}

# Multiple negatives ranking (MNR) loss

In [15]:
mnli = train_dataset.filter(lambda x: True if x["label"] == 0 else False)

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
mnli[0]

{'premise': 'you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him',
 'hypothesis': 'You lose the things to the following level if the people recall.',
 'label': 0}

In [17]:
import random
from datasets import Dataset
# Prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = mnli["hypothesis"]
random.shuffle(soft_negatives)
for row, soft_negative in zip(mnli, soft_negatives):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)

In [18]:
# Loss function Bert
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model)

Step,Training Loss
100,0.516900
200,0.102800
300,0.076100
400,0.065900
500,0.068800


{'pearson_cosine': np.float64(0.811510732228347),
 'spearman_cosine': np.float64(0.8152486437376586)}

In [19]:
# Loss function MPNet
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model_two)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model_two",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model_two,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model_two)

Step,Training Loss
100,1.559800
200,0.125200
300,0.096400
400,0.087800
500,0.063900


{'pearson_cosine': np.float64(0.8325786671454314),
 'spearman_cosine': np.float64(0.8343561426720877)}

# Fine-Tuning Pre-trained Embedding Models

## Supervised/Fine-Tuning with all-MiniLM-L6-v2

The pre-trained model [all-MiniLM-L6-v2](https://https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) is based on Microsoft's distilled model [MiniLM](https://huggingface.co/microsoft/MiniLM-L12-H384-uncased) which is a  12-layer encoder based Transformer model that is uncased and produces a 384 dimensional embedding. ALL-MiniLM-L6-v2 is a 6 layer [variant](https://huggingface.co/nreimers/MiniLM-L6-H384-uncased) of Microsoft's pre-trained model.

From a fine-tuning perspective all-MiniLM-L6-v2 was fine-tuned on 1B sentence pairs of the following datasets:

<div class="max-w-full overflow-auto">
<table>
<thead><tr>
<th>Dataset</th>
<th align="center">Paper</th>
<th align="center">Number of training tuples</th>
</tr>
</thead><tbody><tr>
<td><a rel="nofollow" href="https://github.com/PolyAI-LDN/conversational-datasets/tree/master/reddit">Reddit comments (2015-2018)</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1904.06472">paper</a></td>
<td align="center">726,484,430</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> Citation pairs (Abstracts)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">116,288,806</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/afader/oqa#wikianswers-corpus">WikiAnswers</a> Duplicate question pairs</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.1145/2623330.2623677">paper</a></td>
<td align="center">77,427,422</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/facebookresearch/PAQ">PAQ</a> (Question, Answer) pairs</td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/2102.07033">paper</a></td>
<td align="center">64,371,441</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> Citation pairs (Titles)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">52,603,982</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> (Title, Abstract)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">41,769,185</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title, Body) pairs</td>
<td align="center">-</td>
<td align="center">25,316,456</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title+Body, Answer) pairs</td>
<td align="center">-</td>
<td align="center">21,396,559</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title, Answer) pairs</td>
<td align="center">-</td>
<td align="center">21,396,559</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://microsoft.github.io/msmarco/">MS MARCO</a> triplets</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.1145/3404835.3462804">paper</a></td>
<td align="center">9,144,553</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/gooaq">GOOAQ: Open Question Answering with Diverse Answer Types</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/pdf/2104.08727.pdf">paper</a></td>
<td align="center">3,012,496</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Title, Answer)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">1,198,260</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/code_search_net">Code Search</a></td>
<td align="center">-</td>
<td align="center">1,151,414</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://cocodataset.org/#home">COCO</a> Image captions</td>
<td align="center"><a rel="nofollow" href="https://link.springer.com/chapter/10.1007%2F978-3-319-10602-1_48">paper</a></td>
<td align="center">828,395</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/specter">SPECTER</a> citation triplets</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/2020.acl-main.207">paper</a></td>
<td align="center">684,100</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Question, Answer)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">681,164</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Title, Question)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">659,896</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/search_qa">SearchQA</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1704.05179">paper</a></td>
<td align="center">582,261</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/eli5">Eli5</a></td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/p19-1346">paper</a></td>
<td align="center">325,475</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://shannon.cs.illinois.edu/DenotationGraph/">Flickr 30k</a></td>
<td align="center"><a rel="nofollow" href="https://transacl.org/ojs/index.php/tacl/article/view/229/33">paper</a></td>
<td align="center">317,695</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (titles)</td>
<td align="center"></td>
<td align="center">304,525</td>
</tr>
<tr>
<td>AllNLI (<a rel="nofollow" href="https://nlp.stanford.edu/projects/snli/">SNLI</a> and <a rel="nofollow" href="https://cims.nyu.edu/~sbowman/multinli/">MultiNLI</a></td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/d15-1075">paper SNLI</a>, <a rel="nofollow" href="https://doi.org/10.18653/v1/n18-1101">paper MultiNLI</a></td>
<td align="center">277,230</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (bodies)</td>
<td align="center"></td>
<td align="center">250,519</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (titles+bodies)</td>
<td align="center"></td>
<td align="center">250,460</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/google-research-datasets/sentence-compression">Sentence Compression</a></td>
<td align="center"><a rel="nofollow" href="https://www.aclweb.org/anthology/D13-1155/">paper</a></td>
<td align="center">180,000</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/pvl/wikihow_pairs_dataset">Wikihow</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1810.09305">paper</a></td>
<td align="center">128,542</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/chridey/altlex/">Altlex</a></td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/P16-1135.pdf">paper</a></td>
<td align="center">112,696</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://quoradata.quora.com/First-Quora-Dataset-Release-Question-Pairs">Quora Question Triplets</a></td>
<td align="center">-</td>
<td align="center">103,663</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://cs.pomona.edu/~dkauchak/simplification/">Simple Wikipedia</a></td>
<td align="center"><a rel="nofollow" href="https://www.aclweb.org/anthology/P11-2117/">paper</a></td>
<td align="center">102,225</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://ai.google.com/research/NaturalQuestions">Natural Questions (NQ)</a></td>
<td align="center"><a rel="nofollow" href="https://transacl.org/ojs/index.php/tacl/article/view/1455">paper</a></td>
<td align="center">100,231</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://rajpurkar.github.io/SQuAD-explorer/">SQuAD2.0</a></td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/P18-2124.pdf">paper</a></td>
<td align="center">87,599</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/trivia_qa">TriviaQA</a></td>
<td align="center">-</td>
<td align="center">73,346</td>
</tr>
<tr>
<td><strong>Total</strong></td>
<td align="center"></td>
<td align="center"><strong>1,170,060,424</strong></td>
</tr>
</tbody>
</table>
</div>

fine-tuning was based using a contrastive learning objective, in which the cosine similarity from each possible sentence pair from the batch was computed and then the cross entropy loss was computed by comparing it with the true pair. As detailed by the [authors](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) the model is intended to be used as a sentence and short paragraph encoder. Given an input text, it outputs a vector which captures the semantic information. The sentence vector may be used for information retrieval, clustering or sentence similarity tasks.





In [22]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
train_dataset_three = train_dataset
train_dataset_three

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 16875
})

In [23]:
# Create an embedding similarity evaluator for stsb
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [24]:
# Define model
fine_embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=fine_embedding_model)
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=fine_embedding_model,
    args=args,
    train_dataset=train_dataset_three,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()
# Evaluate our trained model
evaluator(embedding_model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.038700
200,0.042800
300,0.044700
400,0.040000
500,0.044300


{'pearson_cosine': np.float64(0.811510732228347),
 'spearman_cosine': np.float64(0.8152486437376586)}